In [1]:
import sys
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm

In [2]:
sys.path.append("/home/alexis/Documents/COMP558/COMP558-FinalProject/code/")

In [3]:
from optical_flow.bounding_box import BoundingBox
from feature_description.orb_descriptor import ORBDescriptor
from optical_flow.gu import Gu

In [4]:
# video_name = "videos/VIDEO-20250109-160653.mp4"
# depth_name = "videos/DEPTH-20250109-160653.mp4"
video_name = "/home/alexis/Documents/master/vision/repo/libfreenect/wrappers/python/out/VIDEO-20250113-190349.mp4"
depth_name = "/home/alexis/Documents/master/vision/repo/libfreenect/wrappers/python/out/DEPTH-20250113-190349.mp4"

output_name = "out/gu_box.mp4"

In [5]:
x, y, w, h = 275, 200, 110, 85

orb_params = {"params": {"nfeatures" : 10000, "edgeThreshold" : 5, "patchSize" : 5}}

In [6]:
initial_bbox = BoundingBox(x, y, w, h)

feature_descriptor = ORBDescriptor(**orb_params)

In [7]:
cap_video = cv.VideoCapture(video_name)

ret_video, frame_video = cap_video.read()

In [ ]:
fig, ax = plt.subplots()

ax.imshow(frame_video[:,:,::-1])

rect = patches.Rectangle((initial_bbox.x, initial_bbox.y), initial_bbox.w, initial_bbox.h, linewidth=1, edgecolor='r', facecolor='none')
ax.add_patch(rect)

plt.show()

In [ ]:
gu = Gu(frame_video, initial_bbox, feature_descriptor, scale_factor=1, frame_buffer=60, gamma=0.1)

In [10]:
fps = cap_video.get(cv.CAP_PROP_FPS)

fourcc = cv.VideoWriter_fourcc(*'mp4v')
video_writer = cv.VideoWriter(output_name, fourcc, fps, frame_video.shape[:-1][::-1])

In [ ]:
length = int(cap_video.get(cv.CAP_PROP_FRAME_COUNT))

# while ret_video and ret_depth:
for i in tqdm(range(length)):

    if not ret_video:
        break
    

    points_loc, points_desc, points_size = feature_descriptor.detect_features(frame_video)


    bbox, _ = gu.track_frame(frame_video)

    img2 = cv.rectangle(frame_video, (bbox.x, bbox.y), (bbox.x + bbox.w, bbox.y + bbox.h), 255, 2)
    
    for j in range(points_loc.shape[0]):
        img2 = cv.circle(img2, np.int_(points_loc[j]), 1, (0,0,255), -1)

    video_writer.write(img2)

    ret_video, frame_video = cap_video.read()

In [12]:
video_writer.release()